# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [115]:
###############################################################
# SETUP — MODERN VERSION COMPATIBLE WITH LANGCHAIN 2026
###############################################################

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
import tiktoken
from IPython.display import Markdown, display
from dotenv import load_dotenv

load_dotenv("../../")

# GPT-4o = MODERN AGENT
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# TOKENISER FOR TRUNCATING TAVILY
encoding = tiktoken.encoding_for_model("gpt-4o")

# TAVILY INSTANCE
tavily_search = TavilySearchResults()

###############################################################
# SAFE SEARCH TOOL — MODERN VERSION
###############################################################

def TavilySafeSearch(query: str) -> str:
    """
    TAVILY WEB SEARCH + TRIMMING TO AVOID TOKEN OVERFLOW.
    """
    result = tavily_search.run(query)

    # Extract text from Tavily result
    if isinstance(result, dict):
        text = (
            result.get("content")
            or result.get("snippet")
            or result.get("snippets")
            or str(result)
        )

        # If snippets is a list → join
        if isinstance(text, list):
            text = " ".join(map(str, text))

        # Ensure text is a string
        text = str(text)

        # Truncate to avoid token overflow
        tokens = encoding.encode(text)
        trimmed = encoding.decode(tokens[:3500])
        return trimmed

    # Fallback
    return str(result)


# TOOLS LIST
tools = [
    {
        "type": "function",
        "function": {
            "name": "TavilySafeSearch",
            "description": "Retrieve safe web search results using Tavily",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"]
            }
        }
    }
]


###############################################################
# AGENT MODERNE — GPT‑4o TOOL-CALLING AUTOMATIQUE 
###############################################################

def agent(query: str) :
#    tool_call = []
#    name = []
#    args = []
    msg = [{"role": "user", "content": query}]
#    if msg.tool_calls:
#        tool_call = msg.tool_calls[0]
#        name = tool_call["name"]
#        args = tool_call["args"]
    
    while True:
        # Step 1: model responds
        response = llm.invoke(msg, tools=tools)

        # If no tool calls → final answer
        if not response.tool_calls:
            return response.content

        msg.append({
            "role": "assistant",
            "content": response.content,
            "tool_calls": response.tool_calls
        })

        # Step 2: execute each tool call
        for call in response.tool_calls:
            name = call["name"]
            args = call["args"]

            if name == "TavilySafeSearch":
                result = TavilySafeSearch(**args)
            else:
                result = f"Unknown tool: {name}"

            # Step 3: append tool result message
            msg.append({
                "role": "tool",
                "tool_call_id": call["id"],
                "content": result
            })

        # Step 4: append assistant tool call message
#        msg.append(response)


### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [116]:
# YOUR PROMT HERE
prompt_1 = """
You MUST call the tool TavilySafeSearch.
Do NOT answer directly.

Call TavilySafeSearch with:
{
  "query": "recent advancements in generative AI applied to healthcare 2025"
}

After receiving the tool result, produce a clean Markdown summary with:
- Key breakthroughs
- Companies involved
- Real-world applications
- Sources
"""

# RUN IT
response_1 = agent(prompt_1)


In [117]:
type(response_1)

str

In [118]:
display(Markdown(response_1))

### Recent Advancements in Generative AI Applied to Healthcare (2025)

#### Key Breakthroughs
- **Enhanced Diagnostic Accuracy**: Generative AI systems are improving diagnostic precision across various medical specialties, leading to better patient outcomes and reduced medical errors.
- **Drug Discovery and Development**: AI is accelerating drug discovery by predicting and creating novel drug compounds, significantly reducing the time and cost of bringing new medications to market.
- **AI-Powered Imaging and Diagnostic Triage**: AI-assisted imaging workflows are improving cancer detection rates and reducing unnecessary recalls, enhancing clinical capacity.
- **Multimodal AI Models**: The shift towards multimodal models capable of analyzing text, images, genomics data, and real-time patient vitals is enhancing diagnostics and clinical decision support.

#### Companies Involved
- **Health IT Vendors**: Collaborating with AI developers to integrate AI into clinical workflows.
- **Pharmaceutical Companies**: Utilizing AI for drug discovery and development.

#### Real-World Applications
- **Clinical Workflows**: AI is reducing clinician documentation burdens and enhancing patient communication.
- **Operational Efficiencies**: AI is streamlining clinician workflows, optimizing resource utilization, and reducing administrative burdens.
- **Functional and Integrative Medicine**: AI models analyze patient history and genetic data to provide comprehensive treatment recommendations.

#### Sources
1. [Top 10 Use Cases Of Generative AI In Healthcare For 2025](https://murphi.ai/generative-ai-in-healthcare-use-cases-benefits-and-applications)
2. [Generative Artificial Intelligence in Healthcare: Applications](https://www.mdpi.com/2673-7426/5/3/37)
3. [Generative AI in Healthcare: Use Cases, Benefits](https://www.johnsnowlabs.com/generative-ai-healthcare)
4. [Generative AI in clinical (2020–2025): a mini-review of applications](https://www.frontiersin.org/journals/digital-health/articles/10.3389/fdgth.2025.1653369/full)
5. [The 5 AI Breakthroughs Redefining Healthcare in 2025](https://www.linkedin.com/pulse/5-ai-breakthroughs-redefining-healthcare-2025-7shec)

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [119]:
# YOUR PROMPT HERE
prompt_2 = """
Search for the top emerging AI startups in 2025.
Provide a structured Markdown report including:
- Company names
- Their main innovations
- Funding or traction indicators
- Sources
"""
# RUN IT
response_2 = agent(prompt_2)

In [120]:
display(Markdown(response_2))

Here's a structured Markdown report on some of the top emerging AI startups in 2025:

## Emerging AI Startups in 2025

### 1. Ambience Healthcare
- **Main Innovations**: AI operating system for healthcare workflows, focusing on clinical documentation, coding, and workflows to reduce administrative burdens on clinicians.
- **Funding/Traction**: Total funding of $243M (Series C, 2025).
- **Sources**: [Wellows](https://wellows.com/blog/ai-startups)

### 2. Fathom.ai
- **Main Innovations**: AI meeting assistant for productivity and sales.
- **Funding/Traction**: Founded in 2025, with a 20-person team.
- **Sources**: [Wellows](https://wellows.com/blog/ai-startups)

### 3. Kilo
- **Main Innovations**: Open-source agentic engineering platform offering AI coding assistance with access to 500+ models.
- **Funding/Traction**: $8M seed round (December 2025), 420,000+ downloads, and $2.2M revenue in 2025.
- **Sources**: [Wellows](https://wellows.com/blog/ai-startups)

### 4. Thinking Machines Lab
- **Main Innovations**: AI research and products.
- **Funding/Traction**: $2B in funding, founded in 2025.
- **Sources**: [Forbes](https://www.forbes.com/lists/ai50)

### 5. Greptile
- **Main Innovations**: AI code review, enhancing the ability to manage large volumes of code.
- **Funding/Traction**: $180 million valuation after a $25 million Series A (September 2025).
- **Sources**: [Medium](https://medium.com/jonathans-musings/ai-startups-you-should-know-48d69d1ddf61)

### 6. Gaia AI
- **Main Innovations**: AI solutions for biodiversity management and wildfire risk mitigation in forestry.
- **Funding/Traction**: Featured at the 2025 MIT AI Conference.
- **Sources**: [MIT Sloan](https://mitsloan.mit.edu/ideas-made-to-matter/10-mit-ai-startups-to-watch-2025)

These startups are at the forefront of AI innovation, each contributing unique solutions to various industries.

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [121]:
# YOUR PROMPT HERE
prompt_3 = """
Compare OpenAI's ChatGPT Team vs Anthropic's Claude Pro.
Provide a structured Markdown comparison including:
- Features
- Pricing
- Pros & cons
- User reviews
- Sources
"""
# RUN IT
response_3 = agent(prompt_3)

In [122]:
display(Markdown(response_3))


Here's a structured Markdown comparison of OpenAI's ChatGPT Team and Anthropic's Claude Pro based on the retrieved information:

## OpenAI's ChatGPT Team

### Features
- **Natural Language Processing**: Advanced conversation capabilities.
- **Multimodal Capabilities**: Supports text, images, voice, and video.
- **Code Assistance**: Via Codex for generating, refactoring, and debugging code.
- **Collaboration Tools**: Shared workspace, admin controls, and team memory.
- **Security and Compliance**: SOC 2 Type 2, ISO 27001, GDPR, CCPA compliance.
- **Integration**: Connects with tools like Slack, Google Drive, SharePoint, GitHub.

### Pricing
- **Free Plan**: $0/month with limited features.
- **Plus Plan**: $20/user/month.
- **Team Plan**: $25/user/month (annual billing) or $30/user/month (monthly billing).
- **Pro Plan**: $200/user/month for power users.

### Pros & Cons
**Pros**:
- Comprehensive feature set.
- Strong security and compliance features.
- Good for collaborative workflows.

**Cons**:
- Higher cost for premium features.
- Limited enterprise-grade security in lower plans.

### User Reviews
- Generally positive with a 4.3 out of 5 rating.
- Users appreciate the ease of setup and integration.
- Some concerns about cost for smaller teams.

### Sources
- [SalesHive](https://saleshive.com/vendors/chatgpt-openai)
- [PeerSpot](https://www.peerspot.com/products/chatgpt-team-enterprise-reviews)
- [CloudEagle](https://www.cloudeagle.ai/blogs/blog-chatgpt-pricing-guide)

## Anthropic's Claude Pro

### Features
- **Advanced AI Models**: Access to Claude Opus 4.6 and beyond.
- **Task Automation**: Multi-step task automation with Cowork.
- **Code Review**: Agentic coding assistant for codebase management.
- **Integration Support**: Full integration support for various applications.
- **Extended Usage Limits**: Higher usage limits compared to free tier.

### Pricing
- **Free Plan**: Basic access with limited features.
- **Pro Plan**: $20/month or $17/month with annual billing.

### Pros & Cons
**Pros**:
- Slightly cheaper than competitors.
- Strong reasoning and creative writing capabilities.
- User-friendly interface.

**Cons**:
- Trains on user data by default.
- Lacks image and video generation tools.

### User Reviews
- Positive feedback for its reasoning and creative capabilities.
- Criticized for default data training and lack of multimedia tools.

### Sources
- [FlowStep](https://flowstep.ai/blog/claude-code-review)
- [Aionx](https://aionx.co/chatgpt-reviews/claude-pro-features-complete-list)
- [PCMag Australia](https://au.pcmag.com/ai/111916/claude)

This comparison provides an overview of the features, pricing, pros and cons, and user reviews for both ChatGPT Team and Claude Pro.

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.


In [123]:
prompt_bonus = """
Propose a real-world use case where an AI agent using TavilySafeSearch
could provide high value. Then implement the solution using the agent.
Provide a Markdown explanation + the agent’s output.
"""

response_bonus = agent(prompt_bonus)

In [124]:
display(Markdown(response_bonus))

## AI Agent's Output

### Safe Search Results for "Climate Change Educational Resources"

1. **[Climate Education Resources from The Wild Center](https://www.wildcenter.org/our-work/youth-climate-program/resources)**
   - This resource offers a variety of climate education materials, including videos, lesson plans, and interactive activities. It is designed to engage students and teachers with climate change topics through hands-on activities and virtual programs.

2. **[Climate Education Resources - Rutgers Climate and Energy Institute (RCEI)](https://rcei.rutgers.edu/resources/climate-education-resources)**
   - Provides a comprehensive set of resources for different audiences, including K-12 educators. It includes guides, online courses, and interactive tools to support climate science education.

3. **[A Component-Based Approach to Open Educational Resources in Climate Change Education](https://www.aacu.org/oer/research/a-component-based-approach-to-open-educational-resources-in-climate-change-education)**
   - This initiative offers open educational resources (OER) for climate change education, including photographs, videos, and presentations, which can be customized for various educational contexts.

4. **[Climate Education | OER Commons](https://oercommons.org/hubs/climate)**
   - A library of educational resources about climate change, available under a copyright license that allows free sharing and customization. It aims to provide equitable access to climate education materials.

5. **[Teaching about Climate Change - Hennepin County](https://www.hennepincounty.gov/climate-action/what-we-can-do/teaching-climate)**
   - Offers a range of educational resources, including curriculum programs, activities, and digital tools, to help educators teach climate change effectively.

### Explanation

The AI agent successfully retrieved a list of safe and reliable educational resources on climate change. These resources are suitable for educators looking to enhance their lesson plans with accurate and engaging content. By using TavilySafeSearch, the AI agent ensures that the information is appropriate for educational settings, thus providing high value to teachers and students alike.